In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

CUDA available: True
GPU device: NVIDIA A40
Number of GPUs: 1


In [3]:
# Set device for GPU compute
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Define paths
original_repo_path = '/net/scratch2/smallyan/relations_eval'
replication_path = '/net/scratch2/smallyan/relations_eval/evaluation/replications'
output_path = '/net/scratch2/smallyan/relations_eval/evaluation/new_replication_eval'

# Check if paths exist
print(f"Original repo exists: {os.path.exists(original_repo_path)}")
print(f"Replication path exists: {os.path.exists(replication_path)}")

# List contents
print("\nOriginal repo contents:")
if os.path.exists(original_repo_path):
    for item in os.listdir(original_repo_path):
        print(f"  {item}")

Original repo exists: True
Replication path exists: True

Original repo contents:
  evaluation
  hparams
  pyproject.toml
  data
  experiments.py
  __MACOSX
  CodeWalkthrough.md
  plan.md
  no_exe_evaluation
  notebooks
  lre_cached
  src
  requirements.txt
  schematic-wide.png
  invoke.yaml
  lre_cached.zip
  LICENSE
  .gitignore
  tests
  __pycache__
  doc_only_evaluation
  results
  .git
  demo
  tasks.py
  scripts
  documentation.pdf


In [5]:
# Look for documentation files in original repo
print("Looking for documentation files in original repo:")
for root, dirs, files in os.walk(original_repo_path):
    for f in files:
        if 'documentation' in f.lower() or 'readme' in f.lower() or f.endswith('.md'):
            rel_path = os.path.relpath(os.path.join(root, f), original_repo_path)
            print(f"  {rel_path}")

Looking for documentation files in original repo:
  CodeWalkthrough.md
  plan.md
  documentation.pdf
  evaluation/replications/evaluation_replication.md
  evaluation/replications/documentation_replication.md
  evaluation/replication_eval/documentation_evaluation_summary.md
  evaluation/replication_eval/documentation_eval_summary.json
  no_exe_evaluation/replications/no_exe_evaluation_replication.md
  doc_only_evaluation/replication_evaluation.md


In [6]:
# List replication path contents
print("Replication path contents:")
if os.path.exists(replication_path):
    for item in os.listdir(replication_path):
        print(f"  {item}")

Replication path contents:
  replication_output.txt
  evaluation_replication.md
  documentation_replication.md
  run_replication.py
  replication.ipynb
  self_replication_evaluation.json
  replication_results.json


In [7]:
# Read the original documentation - try CodeWalkthrough.md first as the main documentation
original_doc_path = os.path.join(original_repo_path, 'CodeWalkthrough.md')
with open(original_doc_path, 'r') as f:
    original_doc = f.read()
print("=" * 80)
print("ORIGINAL DOCUMENTATION (CodeWalkthrough.md)")
print("=" * 80)
print(original_doc[:5000])

ORIGINAL DOCUMENTATION (CodeWalkthrough.md)
# Linearity of Relation Decoding in Transformer LMs

This repository contains the code used for the experiments in the paper [Linearity of Relation Decoding in Transformer LMs](https://arxiv.org/abs/2308.09124). 

How an LM decodes a relation such as (*Miles Davis*, **plays the instrument**, *trumpet*) involves a sequence of non-linear computations spanning multiple layers. But, in this work we show that for a subset of relations this highly non-linear decoding procedure can be approximated by a simple *linear transformation* ($\textbf{LRE}$) on the subject representation $\mathbf{s}$ at some intermediate layer. 

<p align="center">
<img src="schematic-wide.png" style="width:80%;"/>
</p>

Please check [lre.baulab.info](https://lre.baulab.info/) for more information.



## Setup

All code is tested on `MacOS Ventura (>= 13.1)` and `Ubuntu 20.04` using `Python >= 3.10`. It uses a lot of newer Python features, so the Python version is a strict r

In [8]:
# Let's also check the demo notebook for specific results to compare
demo_path = os.path.join(original_repo_path, 'demo')
print("Demo folder contents:")
if os.path.exists(demo_path):
    for item in os.listdir(demo_path):
        print(f"  {item}")

Demo folder contents:
  attribute_lens.ipynb
  demo.ipynb
  attribute_lens.png
  causality-crop.png


In [9]:
# Read the demo notebook to understand the expected results
import json

demo_notebook_path = os.path.join(original_repo_path, 'demo', 'demo.ipynb')
with open(demo_notebook_path, 'r') as f:
    demo_notebook = json.load(f)

# Extract cells and their outputs
print("Demo notebook cell analysis:")
print("=" * 80)
for i, cell in enumerate(demo_notebook['cells']):
    if cell['cell_type'] == 'markdown':
        content = ''.join(cell['source'])[:200]
        print(f"\n[Cell {i}] MARKDOWN:")
        print(content[:200])
    elif cell['cell_type'] == 'code':
        source = ''.join(cell['source'])[:150]
        print(f"\n[Cell {i}] CODE: {source[:150]}...")

Demo notebook cell analysis:

[Cell 0] CODE: import sys
sys.path.append('..')

import torch
from src import models, data, lens, functional
from src.utils import experiment_utils
from baukit impor...

[Cell 1] CODE: device = "cuda:0"
mt = models.load_model("gptj", device=device, fp16=True)
print(f"dtype: {mt.model.dtype}, device: {mt.model.device}, memory: {mt.mod...

[Cell 2] CODE: dataset = data.load_dataset()

relation_names = [r.name for r in dataset.relations]
relation_options = Menu(choices = relation_names, value = relation...

[Cell 3] CODE: relation_name = relation_options.value
relation = dataset.filter(relation_names=[relation_name])[0]
print(f"{relation.name} -- {len(relation.samples)}...

[Cell 4] CODE: ################### hparams ###################
layer = 5
beta = 2.5
###############################################...

[Cell 5] CODE: from src.operators import JacobianIclMeanEstimator

estimator = JacobianIclMeanEstimator(
    mt = mt, 
    h_layer = layer,
    beta = bet

In [10]:
# Let's read the demo notebook outputs to find specific numerical results
for i, cell in enumerate(demo_notebook['cells']):
    if cell['cell_type'] == 'code' and 'outputs' in cell and cell['outputs']:
        source = ''.join(cell['source'])[:100]
        print(f"\n[Cell {i}] CODE: {source}...")
        print("OUTPUTS:")
        for output in cell['outputs']:
            if 'text' in output:
                print(''.join(output['text'])[:500])
            elif 'data' in output:
                if 'text/plain' in output['data']:
                    print(''.join(output['data']['text/plain'])[:500])


[Cell 1] CODE: device = "cuda:0"
mt = models.load_model("gptj", device=device, fp16=True)
print(f"dtype: {mt.model....
OUTPUTS:
dtype: torch.float16, device: cuda:0, memory: 12219206136


[Cell 2] CODE: dataset = data.load_dataset()

relation_names = [r.name for r in dataset.relations]
relation_options...
OUTPUTS:

[Cell 3] CODE: relation_name = relation_options.value
relation = dataset.filter(relation_names=[relation_name])[0]
...
OUTPUTS:
country capital city -- 24 samples
------------------------------------------------------
China -> Beijing
Japan -> Tokyo
Italy -> Rome
Brazil -> Bras\u00edlia
Turkey -> Ankara


[Cell 5] CODE: from src.operators import JacobianIclMeanEstimator

estimator = JacobianIclMeanEstimator(
    mt = m...
OUTPUTS:
relation has > 1 prompt_templates, will use first (The capital city of {} is)


[Cell 8] CODE: sample = test.samples[0]
print(sample)
operator(subject = sample.subject).predictions...
OUTPUTS:
Argentina -> Buenos Aires

[PredictedToken(token=' Bue

In [11]:
# Let's get more specific results from the demo - the last cells with summary metrics
for i, cell in enumerate(demo_notebook['cells']):
    if cell['cell_type'] == 'code' and 'outputs' in cell and cell['outputs']:
        if i >= 23:  # Focus on later cells for summary metrics
            source = ''.join(cell['source'])
            print(f"\n[Cell {i}] FULL CODE:")
            print(source)
            print("\nOUTPUTS:")
            for output in cell['outputs']:
                if 'text' in output:
                    print(''.join(output['text']))
                elif 'data' in output:
                    if 'text/plain' in output['data']:
                        print(''.join(output['data']['text/plain']))


[Cell 23] FULL CODE:
# precomputing latents to speed things up
hs_and_zs = functional.compute_hs_and_zs(
    mt = mt,
    prompt_template = operator.prompt_template,
    subjects = [sample.subject for sample in test.samples],
    h_layer= operator.h_layer,
    z_layer=-1,
    batch_size = 2
)

success = 0
fails = 0

for sample in test.samples:
    target = test_targets.get(sample)
    assert target is not None
    edit_result = editor(
        subject = sample.subject,
        target = target.subject
    )
    
    success_flag = functional.is_nontrivial_prefix(
        prediction=edit_result.predicted_tokens[0].token, target=target.object
    )
    
    print(f"Mapping {sample.subject} -> {target.object} | edit result={edit_result.predicted_tokens[0]} | success=({functional.get_tick_marker(success_flag)})")
    
    success += success_flag
    fails += not success_flag
    
causality = success / (success + fails)

print("------------------------------------------------------------")


In [12]:
# Get faithfulness results from earlier cells
for i, cell in enumerate(demo_notebook['cells']):
    if cell['cell_type'] == 'code' and 'outputs' in cell and cell['outputs']:
        if i == 12:  # Faithfulness cell
            source = ''.join(cell['source'])
            print(f"\n[Cell {i}] FULL CODE:")
            print(source)
            print("\nOUTPUTS:")
            for output in cell['outputs']:
                if 'text' in output:
                    print(''.join(output['text']))
                elif 'data' in output:
                    if 'text/plain' in output['data']:
                        print(''.join(output['data']['text/plain']))


[Cell 12] FULL CODE:
correct = 0
wrong = 0
for sample in test.samples:
    predictions = operator(subject = sample.subject).predictions
    known_flag = functional.is_nontrivial_prefix(
        prediction=predictions[0].token, target=sample.object
    )
    print(f"{sample.subject=}, {sample.object=}, ", end="")
    print(f'predicted="{functional.format_whitespace(predictions[0].token)}", (p={predictions[0].prob}), known=({functional.get_tick_marker(known_flag)})')
    
    correct += known_flag
    wrong += not known_flag
    
faithfulness = correct/(correct + wrong)

print("------------------------------------------------------------")
print(f"Faithfulness (@1) = {faithfulness}")
print("------------------------------------------------------------")

OUTPUTS:
sample.subject='Argentina', sample.object='Buenos Aires', predicted=" Buenos", (p=0.8915027976036072), known=(✓)
sample.subject='Australia', sample.object='Canberra', predicted=" Canberra", (p=0.6969543695449829), known=(✓)
samp

In [13]:
# Now let's read the replicated documentation
replicated_doc_path = os.path.join(replication_path, 'documentation_replication.md')
with open(replicated_doc_path, 'r') as f:
    replicated_doc = f.read()
print("=" * 80)
print("REPLICATED DOCUMENTATION (documentation_replication.md)")
print("=" * 80)
print(replicated_doc)

REPLICATED DOCUMENTATION (documentation_replication.md)
# Documentation: Replication of "Linearity of Relation Decoding in Transformer LMs"

## Goal

Replicate the core experiment from "Linearity of Relation Decoding in Transformer Language Models" (Hernandez et al., 2023), which investigates whether relation decoding in transformer LMs can be approximated by linear transformations on subject representations.

The key hypotheses being tested:
1. For certain relations, the decoding procedure from subject `s` to object `o` can be approximated as an affine transformation: `LRE(s) = W*s + b`
2. This Linear Relational Embedding (LRE) can be computed from the model's Jacobian
3. LRE-based edits to subject representations can successfully change model predictions

## Data

### Dataset Structure
- **Source**: Repository's `data/` directory containing 47 relations across categories:
  - Factual (e.g., country capital, company CEO)
  - Commonsense
  - Linguistic
  - Bias

### Relation Used for R

In [14]:
# Summary of key findings for comparison
print("=" * 80)
print("COMPARISON SUMMARY")
print("=" * 80)

print("\n## ORIGINAL DEMO RESULTS (from demo/demo.ipynb)")
print("-" * 60)
print("Relation: country capital city")
print("Test samples: 19")
print("Faithfulness (@1): 0.7894736842105263 (15/19 correct)")
print("Causality (@1): 1.0 (19/19 successful edits)")
print("Hyperparameters: layer=5, beta=2.5, rank=100")

print("\n## REPLICATED RESULTS (from documentation_replication.md)")
print("-" * 60)
print("Relation: country capital city")
print("Test samples: 19")
print("Faithfulness: 26.32% (5/19 correct)")
print("Causality: 100.00% (19/19 successful edits)")
print("Hyperparameters: layer=5, beta=2.5, rank=100")

print("\n## COMPARISON")
print("-" * 60)
original_faithfulness = 0.7894736842105263
replicated_faithfulness = 0.2632
deviation_faithfulness = abs(original_faithfulness - replicated_faithfulness) / original_faithfulness * 100

original_causality = 1.0
replicated_causality = 1.0
deviation_causality = abs(original_causality - replicated_causality) / original_causality * 100

print(f"Faithfulness: Original={original_faithfulness:.4f}, Replicated={replicated_faithfulness:.4f}")
print(f"  Deviation: {deviation_faithfulness:.2f}%")
print(f"  Within 5% tolerance: {'YES' if deviation_faithfulness <= 5 else 'NO'}")

print(f"\nCausality: Original={original_causality:.4f}, Replicated={replicated_causality:.4f}")
print(f"  Deviation: {deviation_causality:.2f}%")
print(f"  Within 5% tolerance: {'YES' if deviation_causality <= 5 else 'NO'}")

COMPARISON SUMMARY

## ORIGINAL DEMO RESULTS (from demo/demo.ipynb)
------------------------------------------------------------
Relation: country capital city
Test samples: 19
Faithfulness (@1): 0.7894736842105263 (15/19 correct)
Causality (@1): 1.0 (19/19 successful edits)
Hyperparameters: layer=5, beta=2.5, rank=100

## REPLICATED RESULTS (from documentation_replication.md)
------------------------------------------------------------
Relation: country capital city
Test samples: 19
Faithfulness: 26.32% (5/19 correct)
Causality: 100.00% (19/19 successful edits)
Hyperparameters: layer=5, beta=2.5, rank=100

## COMPARISON
------------------------------------------------------------
Faithfulness: Original=0.7895, Replicated=0.2632
  Deviation: 66.66%
  Within 5% tolerance: NO

Causality: Original=1.0000, Replicated=1.0000
  Deviation: 0.00%
  Within 5% tolerance: YES


In [15]:
# Let's also check the replication notebook/output to understand what exactly happened
replication_output_path = os.path.join(replication_path, 'replication_output.txt')
if os.path.exists(replication_output_path):
    with open(replication_output_path, 'r') as f:
        replication_output = f.read()
    print("=" * 80)
    print("REPLICATION OUTPUT (replication_output.txt)")
    print("=" * 80)
    print(replication_output[:5000])

REPLICATION OUTPUT (replication_output.txt)
Using device: cuda:0
CUDA available: True
GPU: NVIDIA H100 NVL

REPLICATION: Linearity of Relation Decoding in Transformer LMs
/home/smallyan/.local/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(

[1/7] Loading GPT-J model...
`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.at

In [16]:
# Let's see more of the replication output
print(replication_output[5000:10000])

cted: '\n' (p=0.263)
----------------------------------------------------------------------
Faithfulness: 26.32% (5/19)

[7/7] Evaluating causality...
Generated 19 edit targets
----------------------------------------------------------------------
[Y] Argentina -> Riyadh, predicted: ' Riyadh' (p=0.739)
[Y] Australia -> Buenos Aires, predicted: ' Buenos' (p=0.899)
[Y] Canada -> Abuja, predicted: ' Abu' (p=0.693)
[Y] Chile -> Lima, predicted: ' Lima' (p=0.766)
[Y] Colombia -> Berlin, predicted: ' Berlin' (p=0.970)
[Y] Egypt -> Mexico City, predicted: ' Mexico' (p=0.975)
[Y] France -> Riyadh, predicted: ' Riyadh' (p=0.753)
[Y] Germany -> Cairo, predicted: ' Cairo' (p=0.948)
[Y] India -> Lima, predicted: ' Lima' (p=0.645)
[Y] Mexico -> Santiago, predicted: ' Santiago' (p=0.848)
[Y] Nigeria -> Riyadh, predicted: ' Riyadh' (p=0.738)
[Y] Pakistan -> New Delhi, predicted: ' New' (p=0.744)
[Y] Peru -> Caracas, predicted: ' Car' (p=0.274)
[Y] Russia -> Cairo, predicted: ' Cairo' (p=0.966)
[Y] Sa

In [17]:
# Check replication results JSON
replication_results_path = os.path.join(replication_path, 'replication_results.json')
if os.path.exists(replication_results_path):
    with open(replication_results_path, 'r') as f:
        replication_results = json.load(f)
    print("Replication Results JSON:")
    print(json.dumps(replication_results, indent=2))

Replication Results JSON:
{
  "timestamp": "2026-01-11T11:44:03.080756",
  "relation": "country capital city",
  "model": "GPT-J-6B",
  "hyperparameters": {
    "h_layer": 5,
    "beta": 2.5,
    "rank": 100
  },
  "data": {
    "train_samples": 5,
    "test_samples": 19
  },
  "results": {
    "faithfulness": 0.2631578947368421,
    "causality": 1.0
  },
  "reference": {
    "faithfulness": 0.789,
    "causality": 1.0
  },
  "replication_status": {
    "faithfulness_match": false,
    "causality_match": true,
    "overall_success": false
  }
}


In [18]:
# Let's also check the evaluation_replication.md for completeness
eval_replication_path = os.path.join(replication_path, 'evaluation_replication.md')
if os.path.exists(eval_replication_path):
    with open(eval_replication_path, 'r') as f:
        eval_replication = f.read()
    print("=" * 80)
    print("EVALUATION REPLICATION (evaluation_replication.md)")
    print("=" * 80)
    print(eval_replication)

EVALUATION REPLICATION (evaluation_replication.md)
# Evaluation: Replication of "Linearity of Relation Decoding in Transformer LMs"

## Reflection

### What Went Well
1. **Environment Setup**: The repository's dependencies were well-documented and the code ran without major modifications
2. **Demo Availability**: The `demo/demo.ipynb` provided a clear reference for the expected workflow and outputs
3. **Code Organization**: The source code was modular and well-structured, making it easy to understand the experimental pipeline
4. **Causality Replication**: The causality metric was perfectly replicated (100%), validating the core hypothesis

### Challenges Encountered
1. **Faithfulness Divergence**: The faithfulness score (26.32%) was significantly lower than the demo reference (78.9%)
2. **Random Variance**: Results depend heavily on the random train/test split, which affects LRE estimation quality
3. **Limited Documentation**: The plan.md describes the overall methodology but lacks spe

In [19]:
# Now let's conduct the formal evaluation based on the criteria
print("=" * 80)
print("DOCUMENTATION EVALUATION - Binary Checklist")
print("=" * 80)

print("""
## DE1. Result Fidelity

**ORIGINAL DEMO RESULTS:**
- Relation: country capital city
- Test samples: 19
- Faithfulness (@1): 78.94% (15/19)
- Causality (@1): 100% (19/19)

**REPLICATED RESULTS:**
- Relation: country capital city
- Test samples: 19
- Faithfulness: 26.32% (5/19)
- Causality: 100% (19/19)

**ANALYSIS:**
- Causality: MATCHES (100% vs 100%, deviation = 0%)
- Faithfulness: DOES NOT MATCH (26.32% vs 78.94%, deviation = 66.66%)

The 5% tolerance threshold is VIOLATED for faithfulness.
The replicated documentation acknowledges this discrepancy and reports it as:
"Faithfulness | 26.32% | Reference (Demo): ~78.9%"

**DE1 VERDICT: FAIL** (Faithfulness result deviates >5% from original)
""")

print("""
## DE2. Conclusion Consistency

**ORIGINAL DOCUMENTATION CONCLUSIONS (CodeWalkthrough.md + Demo):**
1. For certain relations, decoding can be approximated by linear transformation (LRE)
2. LRE can be computed from the model's Jacobian
3. Demo shows both faithfulness (~79%) and causality (100%) as evaluation metrics

**REPLICATED DOCUMENTATION CONCLUSIONS:**
1. "The core hypothesis that relation decoding can be linearized is supported by the causality results"
2. "Causality is robust: Even with lower faithfulness, the LRE's causal structure is preserved"
3. "Linear approximation holds for editing"
4. "Faithfulness is more sensitive: Direct prediction accuracy depends more on training sample selection"

**ANALYSIS:**
The replicated documentation:
- Correctly identifies the core claim (linear approximation of relation decoding)
- Reports causality results consistent with original (100%)
- Acknowledges faithfulness divergence and provides plausible explanations
- HOWEVER, claims partial success when faithfulness differs substantially

The conclusions about the METHOD being valid are consistent, but the conclusion that
the replication "partially succeeds" may be generous given the faithfulness deviation.

**DE2 VERDICT: PASS** (Core scientific conclusions about the methodology are consistent)
""")

print("""
## DE3. No External or Hallucinated Information

**CHECKING FOR EXTERNAL/HALLUCINATED CONTENT:**

1. All relation names, sample data match the original dataset
2. Hyperparameters (layer=5, beta=2.5, rank=100) match the demo
3. Model details (GPT-J-6B) match the original
4. Methodology description matches the original implementation
5. No external references or invented findings are introduced
6. Explanations for divergence are reasonable speculation, clearly labeled as such:
   - "Random Split Variance"
   - "Training Sample Coverage"
   - "Model Version"

**DE3 VERDICT: PASS** (No external or hallucinated information introduced)
""")

print("""
## FINAL VERDICT

| Criterion | Status |
|-----------|--------|
| DE1. Result Fidelity | FAIL |
| DE2. Conclusion Consistency | PASS |
| DE3. No External Information | PASS |

**FINAL DOCUMENTATION VERDICT: REVISION REQUIRED**

The faithfulness result (26.32%) deviates significantly from the original (78.94%),
exceeding the 5% tolerance threshold by a substantial margin (66.66% deviation).
While the causality results match perfectly and the conclusions about methodology
are consistent, the result fidelity criterion is not met.
""")

DOCUMENTATION EVALUATION - Binary Checklist

## DE1. Result Fidelity

**ORIGINAL DEMO RESULTS:**
- Relation: country capital city
- Test samples: 19
- Faithfulness (@1): 78.94% (15/19)
- Causality (@1): 100% (19/19)

**REPLICATED RESULTS:**
- Relation: country capital city
- Test samples: 19
- Faithfulness: 26.32% (5/19)
- Causality: 100% (19/19)

**ANALYSIS:**
- Causality: MATCHES (100% vs 100%, deviation = 0%)
- Faithfulness: DOES NOT MATCH (26.32% vs 78.94%, deviation = 66.66%)

The 5% tolerance threshold is VIOLATED for faithfulness.
The replicated documentation acknowledges this discrepancy and reports it as:
"Faithfulness | 26.32% | Reference (Demo): ~78.9%"

**DE1 VERDICT: FAIL** (Faithfulness result deviates >5% from original)


## DE2. Conclusion Consistency

**ORIGINAL DOCUMENTATION CONCLUSIONS (CodeWalkthrough.md + Demo):**
1. For certain relations, decoding can be approximated by linear transformation (LRE)
2. LRE can be computed from the model's Jacobian
3. Demo shows both

In [20]:
# Create output directory and write evaluation files
import os
from datetime import datetime

# Create output directory
os.makedirs(output_path, exist_ok=True)
print(f"Created/verified output directory: {output_path}")

Created/verified output directory: /net/scratch2/smallyan/relations_eval/evaluation/new_replication_eval


In [21]:
# Write documentation_evaluation_summary.md
evaluation_summary_md = """# Documentation Evaluation Summary

## Evaluation Date
{timestamp}

## Overview
This document evaluates whether the replicator's documentation (`documentation_replication.md`) faithfully reproduces the results and conclusions of the original experiment from the `relations_eval` repository (Linearity of Relation Decoding in Transformer LMs).

---

## Results Comparison

### Original Demo Results (demo/demo.ipynb)
| Metric | Value | Samples |
|--------|-------|---------|
| Faithfulness (@1) | 78.94% | 15/19 |
| Causality (@1) | 100.00% | 19/19 |

### Replicated Results (documentation_replication.md)
| Metric | Value | Samples |
|--------|-------|---------|
| Faithfulness | 26.32% | 5/19 |
| Causality | 100.00% | 19/19 |

### Analysis
- **Causality**: The replicated result (100%) matches the original exactly. This demonstrates that the LRE-based editing mechanism works correctly and reliably redirects model predictions to target objects.
- **Faithfulness**: The replicated result (26.32%) deviates significantly from the original (78.94%), with a deviation of approximately 66.66%. This exceeds the acceptable 5% tolerance threshold.

---

## Conclusions Comparison

### Original Documentation Claims
1. For certain relations, the decoding procedure can be approximated by a linear transformation (LRE)
2. The LRE can be computed from the model's Jacobian
3. Both faithfulness and causality metrics validate the approach

### Replicated Documentation Claims
1. The core hypothesis that relation decoding can be linearized is supported by the causality results
2. Causality is robust even with lower faithfulness
3. Linear approximation holds for editing operations
4. Faithfulness is more sensitive to training sample selection

### Analysis
The replicated documentation maintains consistency with the original's core scientific claims about the validity of linear relation decoding. The conclusions about the methodology being sound are preserved, though the replicated documentation acknowledges a "partial success" due to the faithfulness divergence.

---

## External or Hallucinated Information

No external or hallucinated information was identified in the replicated documentation:
- All relation names and sample data match the original dataset
- Hyperparameters (layer=5, beta=2.5, rank=100) match the demo
- Model details (GPT-J-6B) match the original
- Methodology descriptions match the original implementation
- Explanations for divergence are clearly labeled as speculation/analysis

---

## Evaluation Checklist Summary

| Criterion | Status | Rationale |
|-----------|--------|-----------|
| DE1. Result Fidelity | **FAIL** | Faithfulness (26.32%) deviates 66.66% from original (78.94%), exceeding 5% tolerance. Causality matches exactly. |
| DE2. Conclusion Consistency | **PASS** | Core scientific conclusions about the methodology are consistent with the original. |
| DE3. No External Information | **PASS** | No hallucinated or external information introduced. |

---

## Final Verdict

**REVISION REQUIRED**

The faithfulness result deviates significantly from the original demo, exceeding the 5% tolerance threshold. While the causality results match perfectly and the scientific conclusions about the methodology are consistent, the result fidelity criterion (DE1) is not satisfied.

### Recommendations for Revision
1. Investigate the cause of faithfulness divergence more thoroughly
2. Attempt to replicate using the same random seed and data split as the original demo
3. If divergence persists, document whether this is due to model weight differences or other factors
4. Consider running multiple trials to establish variance bounds
""".format(timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Write the file
summary_md_path = os.path.join(output_path, 'documentation_evaluation_summary.md')
with open(summary_md_path, 'w') as f:
    f.write(evaluation_summary_md)
print(f"Written: {summary_md_path}")

Written: /net/scratch2/smallyan/relations_eval/evaluation/new_replication_eval/documentation_evaluation_summary.md


In [22]:
# Write documentation_eval_summary.json
import json

evaluation_json = {
    "Checklist": {
        "DE1_ResultFidelity": "FAIL",
        "DE2_ConclusionConsistency": "PASS",
        "DE3_NoExternalInformation": "PASS"
    },
    "Rationale": {
        "DE1_ResultFidelity": "The faithfulness metric in the replicated documentation (26.32%) deviates 66.66% from the original demo result (78.94%), significantly exceeding the 5% tolerance threshold. While the causality metric matches exactly (100% vs 100%), the overall result fidelity criterion fails due to the faithfulness discrepancy.",
        "DE2_ConclusionConsistency": "The replicated documentation maintains consistency with the original's core scientific claims. Both documents conclude that relation decoding can be approximated by linear transformations (LRE). The replicated documentation correctly identifies that causality results support this hypothesis and acknowledges that faithfulness is more sensitive to experimental variations.",
        "DE3_NoExternalInformation": "No external or hallucinated information was introduced in the replicated documentation. All data, hyperparameters, and methodology descriptions match the original repository. Explanations for result divergence are clearly labeled as analysis/speculation rather than established facts."
    }
}

# Write the JSON file
summary_json_path = os.path.join(output_path, 'documentation_eval_summary.json')
with open(summary_json_path, 'w') as f:
    json.dump(evaluation_json, f, indent=2)
print(f"Written: {summary_json_path}")

# Display the JSON
print("\nJSON Contents:")
print(json.dumps(evaluation_json, indent=2))

Written: /net/scratch2/smallyan/relations_eval/evaluation/new_replication_eval/documentation_eval_summary.json

JSON Contents:
{
  "Checklist": {
    "DE1_ResultFidelity": "FAIL",
    "DE2_ConclusionConsistency": "PASS",
    "DE3_NoExternalInformation": "PASS"
  },
  "Rationale": {
    "DE1_ResultFidelity": "The faithfulness metric in the replicated documentation (26.32%) deviates 66.66% from the original demo result (78.94%), significantly exceeding the 5% tolerance threshold. While the causality metric matches exactly (100% vs 100%), the overall result fidelity criterion fails due to the faithfulness discrepancy.",
    "DE2_ConclusionConsistency": "The replicated documentation maintains consistency with the original's core scientific claims. Both documents conclude that relation decoding can be approximated by linear transformations (LRE). The replicated documentation correctly identifies that causality results support this hypothesis and acknowledges that faithfulness is more sensit

In [23]:
# Verify the files were written correctly
print("=" * 80)
print("VERIFICATION - Output Files")
print("=" * 80)

print(f"\nOutput directory: {output_path}")
print(f"Directory exists: {os.path.exists(output_path)}")

# List files in output directory
if os.path.exists(output_path):
    files = os.listdir(output_path)
    print(f"\nFiles in output directory:")
    for f in files:
        file_path = os.path.join(output_path, f)
        size = os.path.getsize(file_path)
        print(f"  {f} ({size} bytes)")

# Verify file contents
print("\n" + "=" * 80)
print("File 1: documentation_evaluation_summary.md")
print("=" * 80)
with open(summary_md_path, 'r') as f:
    content = f.read()
print(f"Length: {len(content)} characters")
print("First 500 characters:")
print(content[:500])

print("\n" + "=" * 80)
print("File 2: documentation_eval_summary.json")
print("=" * 80)
with open(summary_json_path, 'r') as f:
    content = json.load(f)
print("JSON structure valid: Yes")
print(f"Checklist keys: {list(content['Checklist'].keys())}")
print(f"Rationale keys: {list(content['Rationale'].keys())}")

VERIFICATION - Output Files

Output directory: /net/scratch2/smallyan/relations_eval/evaluation/new_replication_eval
Directory exists: True

Files in output directory:
  documentation_eval_summary.json (1232 bytes)
  documentation_evaluation_summary.md (3681 bytes)

File 1: documentation_evaluation_summary.md
Length: 3681 characters
First 500 characters:
# Documentation Evaluation Summary

## Evaluation Date
2026-01-16 02:48:25

## Overview
This document evaluates whether the replicator's documentation (`documentation_replication.md`) faithfully reproduces the results and conclusions of the original experiment from the `relations_eval` repository (Linearity of Relation Decoding in Transformer LMs).

---

## Results Comparison

### Original Demo Results (demo/demo.ipynb)
| Metric | Value | Samples |
|--------|-------|---------|
| Faithfulness (@1

File 2: documentation_eval_summary.json
JSON structure valid: Yes
Checklist keys: ['DE1_ResultFidelity', 'DE2_ConclusionConsistency', 'DE3_No

In [24]:
# Final summary
print("=" * 80)
print("REPLICATOR-DOCUMENTATION EVALUATION COMPLETE")
print("=" * 80)

print("""
## Task Summary

Evaluated whether the replicator's documentation faithfully reproduces the 
results and conclusions of the original experiment.

## Input Files
- Original documentation: /net/scratch2/smallyan/relations_eval/CodeWalkthrough.md
- Original demo: /net/scratch2/smallyan/relations_eval/demo/demo.ipynb
- Replicated documentation: /net/scratch2/smallyan/relations_eval/evaluation/replications/documentation_replication.md

## Output Files Created
1. /net/scratch2/smallyan/relations_eval/evaluation/new_replication_eval/documentation_evaluation_summary.md
2. /net/scratch2/smallyan/relations_eval/evaluation/new_replication_eval/documentation_eval_summary.json

## Evaluation Results

| Criterion | Status |
|-----------|--------|
| DE1. Result Fidelity | FAIL |
| DE2. Conclusion Consistency | PASS |
| DE3. No External Information | PASS |

## Final Verdict: REVISION REQUIRED

The faithfulness result (26.32%) deviates significantly (66.66%) from the 
original demo (78.94%), exceeding the 5% tolerance threshold. The causality 
result matches exactly (100%). Conclusions about methodology are consistent,
and no external/hallucinated information was introduced.

## GPU Used: NVIDIA A40 (CUDA available)
""")

print("\nTask completed successfully!")

REPLICATOR-DOCUMENTATION EVALUATION COMPLETE

## Task Summary

Evaluated whether the replicator's documentation faithfully reproduces the 
results and conclusions of the original experiment.

## Input Files
- Original documentation: /net/scratch2/smallyan/relations_eval/CodeWalkthrough.md
- Original demo: /net/scratch2/smallyan/relations_eval/demo/demo.ipynb
- Replicated documentation: /net/scratch2/smallyan/relations_eval/evaluation/replications/documentation_replication.md

## Output Files Created
1. /net/scratch2/smallyan/relations_eval/evaluation/new_replication_eval/documentation_evaluation_summary.md
2. /net/scratch2/smallyan/relations_eval/evaluation/new_replication_eval/documentation_eval_summary.json

## Evaluation Results

| Criterion | Status |
|-----------|--------|
| DE1. Result Fidelity | FAIL |
| DE2. Conclusion Consistency | PASS |
| DE3. No External Information | PASS |

## Final Verdict: REVISION REQUIRED

The faithfulness result (26.32%) deviates significantly (66.66